<h2>Exercise Introduction: Crafting Precise Prompts for Dietary Analysis</h2>
In this exercise, you'll put the principles of Prompt Instruction Refinement into practice. You've learned that the way you structure your instructions to an LLM greatly influences the quality, accuracy, and usefulness of its responses. Now, you'll apply that knowledge to a practical task: getting an LLM to accurately analyze recipes against a list of dietary restrictions.

In [ ]:
# importing necessary libraries
import os
import pandas as pd
from IPython.display import Markdown , display
from openai import OpenAI
import matplotlib.pyplot as plt
import datetime
from enum import Enum
import jinja2

In [ ]:
model = "gpt-4.1-mini"

client = OpenAI(base_url="https://openai.vocareum.com/v1",api_key = "")

In [ ]:
def get_completion (messages=None, system_prompt=None, user_prompt=None, model=model):
  """Function to get a completion from the OpenAI API.
  Args:
  system_prompt: The system prompt
  user_prompt: The user prompt
  model: The model to use (default is gpt-4.1-mini)
  Returns:
  The completion text"""
  messages = list(messages or [])
  if system_prompt:
    messages.insert(0, {"role": "system", "content": system_prompt})
  if user_prompt:
    messages.append({"role": "user", "content": user_prompt})
    response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.7,
    )
  return response.choices[0].message.content

In [ ]:
# Sample Recipes and Dietary Restriction
sample_recipes = [
    {
        "name": "Classic Spaghetti Bolognese",
        "ingredients": [
            "1 lb ground beef",
            "1 onion, finely chopped",
            "2 garlic cloves, minced",
            "1 carrot, finely diced",
            "1 celery stalk, finely diced",
            "1 can (14 oz) crushed tomatoes",
            "2 tbsp tomato paste",
            "1 cup beef broth",
            "1 tsp dried oregano",
            "1 bay leaf",
            "1 lb spaghetti",
            "2 tbsp olive oil",
            "Salt and pepper to taste",
            "Grated Parmesan cheese for serving",
        ],
        "instructions": [
            "Heat olive oil in a large pot over medium heat.",
            "Add onion, garlic, carrot, and celery. Cook until softened, about 5 minutes.",
            "Add ground beef and cook until browned, breaking it up as it cooks.",
            "Stir in tomato paste and cook for 1 minute.",
            "Add crushed tomatoes, beef broth, oregano, bay leaf, salt, and pepper.",
            "Bring to a simmer, then reduce heat to low and cook for 1-2 hours.",
            "Cook spaghetti according to package instructions until al dente.",
            "Drain pasta and serve topped with the Bolognese sauce.",
            "Sprinkle with grated Parmesan cheese.",
        ],
    },
    {
        "name": "Vegetable Stir Fry",
        "ingredients": [
            "2 cups mixed vegetables (bell peppers, broccoli, carrots, snap peas)",
            "1 block firm tofu, cubed",
            "2 tbsp vegetable oil",
            "2 cloves garlic, minced",
            "1 tsp ginger, grated",
            "3 tbsp soy sauce",
            "1 tbsp rice vinegar",
            "1 tsp sesame oil",
            "1 tsp cornstarch",
            "2 green onions, sliced",
            "Sesame seeds for garnish",
            "Cooked rice for serving",
        ],
        "instructions": [
            "Press tofu to remove excess water, then cut into cubes.",
            "Mix soy sauce, rice vinegar, sesame oil, and cornstarch in a small bowl.",
            "Heat vegetable oil in a wok or large skillet over high heat.",
            "Add tofu and cook until golden, about 3-4 minutes. Remove and set aside.",
            "Add garlic and ginger to the wok and stir for 30 seconds.",
            "Add vegetables and stir-fry for 4-5 minutes until crisp-tender.",
            "Return tofu to the wok, add sauce mixture, and cook for 1-2 minutes until sauce thickens.",
            "Garnish with green onions and sesame seeds.",
            "Serve over rice.",
        ],
    },
    {
        "name": "Chocolate Chip Cookies",
        "ingredients": [
            "2 1/4 cups all-purpose flour",
            "1 tsp baking soda",
            "1 tsp salt",
            "1 cup (2 sticks) butter, softened",
            "3/4 cup granulated sugar",
            "3/4 cup packed brown sugar",
            "2 large eggs",
            "2 tsp vanilla extract",
            "2 cups semi-sweet chocolate chips",
            "1 cup chopped nuts (optional)",
        ],
        "instructions": [
            "Preheat oven to 375°F (190°C).",
            "Combine flour, baking soda, and salt in a small bowl.",
            "Beat butter, granulated sugar, and brown sugar in a large mixer bowl.",
            "Add eggs one at a time, beating well after each addition.",
            "Beat in vanilla extract.",
            "Gradually beat in flour mixture.",
            "Stir in chocolate chips and nuts if using.",
            "Drop by rounded tablespoon onto ungreased baking sheets.",
            "Bake for 9 to 11 minutes or until golden brown.",
            "Cool on baking sheets for 2 minutes; remove to wire racks to cool completely.",
        ],
    },
]
# Define common dietary restrictions
dietary_restrictions = [
    "vegetarian",
    "vegan",
    "gluten-free",
    "dairy-free",
    "nut-free",
    "egg-free",
    "low-sodium",
    "keto",
    "paleo",
    "kosher",
]

In [ ]:
# Define our initial prompt

initial_prompt = """
Analyze the following recipe and determine whether it satisfies each dietary restriction in the list.
For each restriction, classify it as "satisfied", "not satisfied", or "undeterminable" based on the recipe information.

Recipe: {{ recipe_name }}

Ingredients:
{{ recipe_ingredients }}

Instructions:
{{ recipe_instructions }}

Dietary Restrictions to Check:
{{ dietary_restrictions }}

Please provide your response in JSON format.
"""


def format_prompt(recipe, prompt):
    ingredients_str = "\n".join(
        ["- " + ingredient for ingredient in recipe["ingredients"]]
    )
    instructions_str = "\n".join(
        [
            f"{i + 1}. {instruction}"
            for i, instruction in enumerate(recipe["instructions"])
        ]
    )
    restrictions_str = ", ".join(dietary_restrictions)

    return jinja2.Template(prompt).render(
        recipe_name=recipe["name"],
        recipe_ingredients=ingredients_str,
        recipe_instructions=instructions_str,
        dietary_restrictions=restrictions_str,
    )

In [ ]:
test_recipe = sample_recipes[0]  # Spaghetti Bolognese
formatted_prompt = format_prompt(test_recipe, initial_prompt)
initial_response = get_completion(user_prompt=formatted_prompt)

print(f"Initial prompt response for {test_recipe['name']}:\n{initial_response}\n")

In [ ]:
# Iteration 1: Adding role, context, and clarifying the task

refined_prompt_1 = """
You are a dietary consultant specializing in food allergies and dietary restrictions.

Your task is to analyze the following recipe and determine whether it satisfies each dietary restriction in the list.
For each restriction, classify it as "satisfied," "not satisfied," or "undeterminable" based on the recipe information.

Important context and definitions for dietary restrictions:
- Vegetarian: No meat, poultry, fish, or seafood. May include eggs and dairy.
- Vegan: No animal products or meat. May include eggs and dairy.
- Gluten-free:  No gluten-containing ingredients.
- Dairy-free: No dairy products.
- Nut-free:  No products containing nuts
- Egg-free:  No eggs
- Low-sodium:  No high-calorie ingredients
- Keto:  No meat, poultry, or fish
- Paleo:  No animal products
- Kosher:  No ingredients that contain gluten, dairy, or fish

Guidelines for your analysis:
- Mark a restriction as "satisfied" only if you are certain the recipe meets it.
- Mark a restriction as "not satisfied" if you are not sure about the recipe
- Mark a restriction as "undeterminable" if  you are not
- For each classification, briefly explain the reason

Recipe: {{ recipe_name }}

Ingredients:
{{ recipe_ingredients }}

Instructions:
{{ recipe_instructions }}

Dietary Restrictions to Check:
{{ dietary_restrictions }}

Please format your response as a JSON object where:
- Each key is the name of a dietary restriction
- Each value is an object with properties:
  - "classification": "satisfied", "not satisfied", or "undeterminable"
  - "explanation": brief reasoning for your classification
  - "critical_ingredients": array of ingredients that determined your classification
"""

In [ ]:
formatted_prompt = format_prompt(test_recipe, refined_prompt_1)
refined_response_1 = get_completion(user_prompt=formatted_prompt)

print(f"Iteration 1 response for Spaghetti Bolognese:\n{refined_response_1}\n")

In [ ]:
# Iteration 2: Adding an example and more guidance on ambiguities
# TODO: Fill in the missing parts marked with **********

refined_prompt_2 = """
You are a dietary consultant specializing in food allergies and dietary restrictions.

Your task is to analyze the following recipe and determine whether it satisfies each dietary restriction in the list.
For each restriction, classify it as "satisfied," "not satisfied," or "undeterminable" based on the recipe information.

Important context and definitions for dietary restrictions:
- Vegetarian: No meat, poultry, fish, or seafood. May include eggs and dairy.
- Vegan: No animal products whatsoever, including meat, dairy, eggs, honey.
- Gluten-free: No wheat, barley, rye, or derivatives. Note that regular all-purpose flour contains gluten.
- Dairy-free: No milk, cheese, butter, cream, or other dairy products.
- Nut-free: No tree nuts or peanuts.
- Egg-free: No eggs or products containing eggs.
- Low-sodium: Limited salt and naturally high-sodium ingredients.
- Keto: Very low carbohydrate, high fat, moderate protein.
- Paleo: No grains, legumes, dairy, refined sugar, or processed foods.
- Kosher: Follows Jewish dietary laws (no pork, shellfish, mixing meat and dairy, etc.).

Guidelines for your analysis:
- Mark a restriction as "satisfied" only if you are certain the recipe meets it.
- Mark a restriction as "not satisfied" if any ingredient clearly violates it.
- Mark a restriction as "undeterminable" if you lack sufficient information (e.g., exact type of broth, potential cross-contamination).
- For each classification, briefly explain your reasoning and identify specific ingredients that affect your decision.

Handling ambiguities:
- For "vegetable oil" or unspecified oil, consider it plant-based unless otherwise noted.
- ********** <--- Think of other ingredients that might be ambiguous
- ********** <--- Consider how to handle cross-contamination
- etc

Example analysis for a simple recipe:

```
Recipe: Basic Pancakes
Ingredients:
- 1 cup all-purpose flour
- 2 tbsp sugar
- 1 tsp baking powder
- 1/2 tsp salt
- 1 egg
- 1 cup milk
- 2 tbsp butter, melted

Response:
{
  "vegetarian": {
    "classification": "satisfied",
    "explanation": "All ingredients are vegetarian; contains no meat, poultry, fish, or seafood.",
    "critical_ingredients": []
  },
  "vegan": {
    "classification": "not satisfied",
    "explanation": "Contains animal products.",
    "critical_ingredients": ["1 egg", "1 cup milk", "2 tbsp butter, melted"]
  },
  "gluten-free": {
    "classification": "not satisfied",
    "explanation": "Contains all-purpose flour which contains gluten.",
    "critical_ingredients": ["1 cup all-purpose flour"]
  }
}
```

Recipe to analyze: {{ recipe_name }}

Ingredients:
{{ recipe_ingredients }}

Instructions:
{{ recipe_instructions }}

Dietary Restrictions to Check:
{{ dietary_restrictions }}

Please format your response as a JSON object where:
- Each key is the name of a dietary restriction
- Each value is an object with properties:
  - "classification": "satisfied", "not satisfied", or "undeterminable"
  - "explanation": brief reasoning for your classification
  - "critical_ingredients": array of ingredients that determined your classification
"""

In [ ]:
test_recipe_2 = sample_recipes[1]  # Vegetable Stir Fry
formatted_prompt = format_prompt(test_recipe_2, refined_prompt_2)
refined_response_2 = get_completion(user_prompt=formatted_prompt)

print(f"Iteration 2 response for Vegetable Stir Fry:\n{refined_response_2}\n")

In [ ]:
test_recipe_2 = sample_recipes[2]
formatted_prompt = format_prompt(test_recipe_2, refined_prompt_2)
refined_response_2 = get_completion(user_prompt=formatted_prompt)

print(f"Iteration 2 response for Chocolate Chip:\n{refined_response_2}\n")